### Install and import dependencies

In [ ]:
!pip install numpy
!pip install lapx
!pip install ultralytics
!pip install roboflow
!pip install opencv-python
!pip install pyyaml 

In [23]:
from ultralytics import YOLO
from roboflow import Roboflow
from pathlib import Path
import cv2
import os

### Establish connections to roboflow workspace and download training dataset

In [21]:

rf = Roboflow(api_key="KoEpvrqTSdXhuxRYoc8h")
project = rf.workspace("huyens-workspace-coh7b").project("animal-detection-bb3te")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Animal-detection-1 in yolov8:: 100%|██████████| 5565/5565 [00:00<00:00, 8260.30it/s]


#### Train model (from beginning)

In [22]:
model = YOLO("yolo11s.pt")

results = model.train(
    data = Path.cwd() / "Animal-detection-1" / "data.yaml",
    epochs=200, imgsz=1280, device = 'mps', batch=2, cache=False,
    amp=False, 
    degrees=180, flipud=0.5, fliplr=0.5,
    patience=50
)

New https://pypi.org/project/ultralytics/8.4.66 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.13.2 torch-2.12.0 MPS (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/alopias/Desktop/Huyen-deePi/Animal-detection-1/data.yaml, degrees=180, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, mult

### Fine-tuning trained model with updated dataset

Download new dataset from roboflow (similar to before but new version of dataset)

In [24]:
rf = Roboflow(api_key="KoEpvrqTSdXhuxRYoc8h")
project = rf.workspace("huyens-workspace-coh7b").project("object-detection-project-fioui")
version = project.version(8)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to object-detection-project-8 in yolov8:: 100%|██████████| 6768/6768 [00:00<00:00, 10754.33it/s]


Direct model to the best trained version

In [31]:
model1 = YOLO("yolo11m.pt")

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11m.pt... <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>


Python(35816) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
######################################################################## 100.0%


In [37]:
results = model1.train(
    data=Path.cwd() / "object-detection-project-8" / "data.yaml", 
    device = 'mps', epochs=2,
    batch=2, freeze=10, cache="disk", amp=False, rect=True, patience=50, workers=4, 
    imgsz=1280,
    lr0=0.001
)

New https://pypi.org/project/ultralytics/8.4.69 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.13.2 torch-2.12.0 MPS (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/alopias/Desktop/Huyen-deePi/object-detection-project-8/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, 

RuntimeError: MPS backend out of memory (MPS allocated: 18.00 GiB, other allocations: 29.62 MiB, max allowed: 18.13 GiB). Tried to allocate 200.00 MiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).